# Exploration des donnees — Terrasses et etalages autorises

Ce notebook explore le jeu de donnees `terrasses-autorisations` (source : Open Data Paris),
qui recense les autorisations d'installation de terrasses et etalages commerciaux sur
l'espace public parisien. L'objectif est de comprendre sa structure avant de le croiser
avec les signalements citoyens de l'application Dans Ma Rue.

In [1]:
import pandas as pd

terrasses = pd.read_csv("../data/terrasses-autorisations.csv", sep=";")
terrasses.shape

(24207, 12)

Le jeu contient 24 207 autorisations de terrasses, une ligne par installation.

Colonnes principales : `typologie` (type d'installation), `adresse`, `arrondissement`,
`nom_enseigne`, `longueur`/`largeur` (dimensions de la terrasse), `periode_installation`
(fenetre saisonniere eventuelle), et les coordonnees geographiques (`geo_point_2d`,
`geo_shape`).

In [1]:
terrasses.info()

<class 'pandas.DataFrame'>
RangeIndex: 24207 entries, 0 to 24206
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   typologie             24197 non-null  str
 1   adresse               24199 non-null  str
 2   arrondissement        23359 non-null  float64
 3   nom_enseigne          23470 non-null  str
 4   nom_societe           734 non-null    str
 5   siret                 23283 non-null  float64
 6   longueur              24191 non-null  float64
 7   largeur               24192 non-null  float64
 8   periode_installation  725 non-null    str
 9   lien_affichette       18764 non-null  str
 10  geo_shape             24207 non-null  str
 11  geo_point_2d          24207 non-null  str
dtypes: float64(4), str(8)
memory usage: 2.2 MB


Deux points a noter avant d'aller plus loin :

- **`arrondissement` est encode comme un code postal** (`75015.0` au lieu de `15`) — a
  harmoniser avant toute jointure avec un autre jeu de donnees qui utiliserait un autre
  codage.
- **`periode_installation` est tres peu renseigne** (725 valeurs sur 24 207, soit 3 %) —
  la grande majorite des terrasses sont donc installees en permanence ou n'ont pas cette
  information documentee.

In [1]:
terrasses['typologie'].value_counts()

typologie
TERRASSES OUVERTES SUR TROTTOIR                        2038
ETALAGE                                                 585
CONTRE TERRASSE ESTIVALE SUR STATIONNEMENT             1256
TERRASSE OUVERTE                                       1783
TERRASSE ESTIVALE SUR TROTTOIR FACE A LA DEVANTURE      405
...                                                     ...
JARDINIERE                                                1
CONTRE ETALAGE                                            2
Name: count, Length: 29, dtype: int64

29 categories de typologie. On note quelques incoherences de graphie probables (par
exemple `TERRASSE FERMEE` sans accent et `TERRASSE FERMÉE` avec accent, qui designent
vraisemblablement la meme categorie) — a garder en tete pour l'analyse qui suit.

In [1]:
terrasses['adresse'].head()

0              31BIS RUE NATIONALE
1                 4 RUE DE LONDRES
2             113 BOULEVARD DAVOUT
3               79 AVENUE DE SEGUR
4       97 RUE DE L AMIRAL MOUCHEZ
Name: adresse, dtype: str

Les adresses sont saisies en **majuscules, sans accents, sans code postal** — un format
different de celui utilise dans Dans Ma Rue (voir notebook suivant), qu'il faudra
harmoniser avant toute jointure.